#!/usr/bin/env python
%matplotlib inline
#
import matplotlib
import numpy as np
import matplotlib.cm as cm
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import math
import plotly
#from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.offline as py
import plotly.graph_objs as go
import pandas as pd
import mpld3    #reguires a separate installation pip install mpld3
from pylab import *
from scipy.special import erf, erfc
from math import pi, log
from scipy import integrate 
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits.mplot3d import Axes3D
py.init_notebook_mode(connected=False)  #initialize offline plotting from notebook

print ('Well Superposition - Zone of Capture')
print ('By N. Sitar, UC Berkeley, 09/2022 \n')
print ('Use consistent units: Time in seconds or days, other units consitent either m or ft \n');
    
# Initialize pdf output
with PdfPages('Zone of Capture.pdf') as pdf:
    matplotlib.rcParams['xtick.direction'] = 'in';
    matplotlib.rcParams['ytick.direction'] = 'in';
    
    # Input
    
    while True:
        try:
            Xmin,Xmax= [float(x) for x in input("Enter Xmin, Xmax: ").split(',')];
            break
        except ValueError:
            print ("Enter comma separated Xmin,Xmax: ")
    while True:
        try:
            Ymin,Ymax= [float(x) for x in input("Enter Ymin, Ymax: ").split(',')];
            break
        except ValueError:
            print ("Enter comma separated Xmin,Xmax: ")
    while True:
        try:
            T=float(input("Transmissivity - T[L**2/T]: "));
            break
        except ValueError:
            print ('Enter T: ');        
    while True:
        try:
            gradient,azimuth=[float(x) for x in input('Regional Gradient [dimensionless], Azimuth from x [degrees]: ').split(',')];
            break
        except ValueError:
            print ("Enter comma separated Gradient, Azimuth [degrees]: ")
    while True:
        try:
            n=int(input('Number of wells (integer)= '));
            Q=[]*(n);
            X1=[]*(n);
            Y1=[]*(n);
            for i in range (n):
                #print(i);
                Q.append(float(input('Pumping rate well %i [L**3/T]: '%(i+1))));
                X1.append(float(input('X-coord, well #%i [L]: '%(i+1))));
                Y1.append(float(input('Y-coord, well #%i [L]: '%(i+1))));
            
            break
        except ValueError:
            print ('Invalid values entered, enter %i comma separated numbers: ')
       
    x=np.arange(Xmin,Xmax,((Xmax-Xmin)/75.)); #create equidistant points in x, increase the value for a finer grid
    y=np.arange(Ymin,Ymax,((Ymax-Ymin)/75.)); #create equidistant points in y
    
    X, Y = np.meshgrid(x, y);               #grid the domain by creating x,y pairs
    az=radians(azimuth);                    # convert degrees to radians
 
 # compute the log r over the grid for each well
    def s(x,y):
        return np.log(np.sqrt((x-X1[i])**2+(y-Y1[i])**2))
    
 # compute the drawdown by each well and superimpose well drawdowns across the grid  
    for i in range (n):
        if i==0:
            S=(Q[i]/(2*pi*T))*s(X,Y);
        else:
            S=S+(Q[i]/(2*pi*T))*s(X,Y);
            
  # superimpose gradient   
    S=S-gradient*(X*cos(az)+Y*sin(az));
    
  # compute velocities for the streamline plot
 
    for i in range (n):
        if i==0:
            ux=-(Q[i]/(2*pi*T)) *(X-X1[i])/((X-X1[i])**2+(Y-Y1[i])**2);
            uy=-(Q[i]/(2*pi*T)) *(Y-Y1[i])/((X-X1[i])**2+(Y-Y1[i])**2);
           
        else:
            ux=ux-(Q[i]/(2*pi*T)) *(X-X1[i])/((X-X1[i])**2+(Y-Y1[i])**2);
            uy=uy-(Q[i]/(2*pi*T)) *(Y-Y1[i])/((X-X1[i])**2+(Y-Y1[i])**2);
  
    # superimpose gradient
    ux=ux+gradient*cos(az);
    uy=uy+gradient*sin(az);
    
 #The commented code below will create regular in line 3D plot of the head distribution
     
    #fig=plt.figure();        
    #ax = plt.axes(projection='3d');
    #ax.plot_surface(X, Y, S,cmap='viridis', edgecolor='none');
    #ax.set_xlabel('X');
    #ax.set_ylabel('Y');
    #ax.set_zlabel('Elevation');
    #plt.show();
    
    # plot streamlines and equipotential lines
    
    width = 10.0
    height = (Ymax - Ymin) / (Xmax - Xmin) * width
    plt.figure(figsize=(width, height))
    plt.xlabel('x', fontsize=16)
    plt.ylabel('y', fontsize=16)
    plt.xlim(Xmin, Xmax)
    plt.ylim(Ymin, Ymax)
    plt.streamplot(x, y, ux, uy,
                  density=1, linewidth=1, arrowsize=2, arrowstyle='->')
    # place markers at the well locations
    plt.scatter(X1, Y1,
               color='#CD2305', s=80, marker='o');         
    CS = plt.contour(X, Y, S,15);
    plt.title('Zone of Capture, Number of Wells = {}, Gradient= {}, Azimuth= {}'.format(n,gradient,azimuth));
    savefig('Zone of Capture.jpg'); #save the streamline plot as a jpg
    pdf.savefig();                  #save the streamline plot as a pdf
    plt.show();
    
    #interactive html plotly code below
    
    fig=go.Figure(go.Surface(x=x,y=y,z=S,contours = {
        "z": {"show": True,"highlight": True,"color":"white", "highlightcolor": "limegreen" }}));
    fig.update_layout(scene = dict(
                    xaxis_title='X',
                    yaxis_title='Y',
                    zaxis_title='Drawdown'),
                    margin=dict(r=20, b=10, l=10, t=10));
    fig.update_layout(xaxis=go.layout.XAxis(title='X-axis'),yaxis=go.layout.YAxis(title='Y'));
    py.offline.plot(fig, filename="Contour.html")
    
    
    plt.show();
    fig.show();
    plt.close();
    
    #stop execution
    #exit(-1);